# DeepExtractor — GravitySpy Classification (O3)

Debug notebook for `scripts/validate_gspy_o3.py`'s GravitySpy re-classification step.
Structure follows `glitchgan/notebooks/gspy_classification.ipynb`, swapping GlitchGAN-generated
signals for DeepExtractor-extracted **residual glitches** (real O3 glitch − DeepExtractor's
reconstructed background). Every individual classification is printed as it happens — this
is meant for interactively debugging why accuracy came out far below chance (~14% for 7
classes) in the CLI run, not for production use.

## Dependencies

Same as `validate_gspy_o3.py` — deepextractor + GravitySpy runtime deps, GravitySpy itself
installed with `--no-deps`, then patched for Python 3.11+/keras 3.x/gwpy 4.x/matplotlib 3.3+/
numpy>=1.24/scikit-image compatibility:

```bash
pip install -e ".[gspy]"
pip install gravityspy==1.0.0 --no-deps
python patch_gspy.py
```

See `patch_gspy.py` (repo root) for exactly what each of the 10 patches does.

In [ ]:
%matplotlib inline

In [1]:
import os

# Must be set before importing tensorflow (via gravityspy) — avoids two issues hit
# running this on CIT: cuDNN init failures on the login node's GPU, and pthread_create()
# failures from TF sizing its CPU thread pool to the whole shared machine (exceeds the
# per-user thread ulimit on login nodes). Harmless to leave set even off-CIT.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import functools
import logging
import pickle
import shutil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from gwdatafind import find_urls
from gwpy.timeseries import TimeSeries
from tqdm.notebook import tqdm

from deepextractor.models.architectures import UNET2D
from deepextractor.utils.checkpoints import load_checkpoint
from deepextractor.utils.mc_dropout import enable_mc_dropout, mc_predict
from deepextractor.utils.stft import apply_istft, apply_stft
from deepextractor.utils.signal import whitened_snr_scaling
from deepextractor.utils.visualization import plot_q_transform

warnings.filterwarnings("ignore")
for _log in ["gwpy", "astropy", "gravityspy", "tensorflow"]:
    logging.getLogger(_log).setLevel(logging.ERROR)

/opt/homebrew/Caskroom/miniforge/base/envs/deepextractor/lib/python3.12/site-packages/gwpy/time/_ligotimegps.py:42: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


## Config

In [2]:
# ── paths — adjust for your checkout ─────────────────────────────────────────
CHECKPOINT   = "/home/tom.dooney/deepextractor/checkpoints_4s_o3_mcd/DeepExtractor_257_checkpoints/checkpoint_best_o3_mcd_tl.pth.tar"
SCALER       = "/home/tom.dooney/deepextractor/data_4s_real/o3_td/scaler_real.pkl"
GSPY_MODEL   = "/home/tom.dooney/glitchgan/models/sidd-cqg-paper-O3-model.h5"
CSV_PATHS    = ["/home/tom.dooney/deepextractor/data_o3a_high_confidence.csv", "/home/tom.dooney/deepextractor/data_o3b_high_confidence.csv"]
OUTPUT_DIR   = "evaluation/gspy_4s_o3_notebook"

# ── run config ────────────────────────────────────────────────────────────────
IFO          = "H1"
DATA_SOURCE  = "cit"        # 'cit' = gwdatafind + local frames (CIT-only, fast); 'open' = GWOSC API
CLEAN_GPS_H1 = 1262540000   # vetted-quiet H1 segment, source: glitchgan/notebooks/gspy_classification.ipynb
N_PER_CLASS  = 3
N_PASSES     = 50           # MC Dropout passes
DROPOUT_P    = 0.1
MIN_SNR      = 15.0
SEED         = 42
RUN_LABEL    = "O3"
SNR_TARGET   = None         # None = inject the residual at its own natural amplitude.
                            # Set a number (e.g. 50, matching glitchgan's diagnostic SNR)
                            # to force-rescale the residual before injection — if that
                            # alone fixes classification, the residual's natural amplitude
                            # was too weak for GravitySpy to recognize, not a pipeline bug.

# ── constants — must match training / get_clean_backgrounds.py ────────────────
SAMPLE_RATE = 4096
LEN_4S = 4 * SAMPLE_RATE
N_FFT, HOP_LENGTH, WIN_LENGTH = 512, 32, 64

FETCH_DUR = 36
MAX_FILTER_DUR = 2
PSD_FFTLENGTH = 4
PSD_OVERLAP = 2
HIGHPASS_HZ = 10.0
MAX_AMP = 30.0

FRAME_TYPE = {"H1": "H1_HOFT_C00", "L1": "L1_HOFT_C00"}

LABEL_ORDER = [
    "Blip", "Fast_Scattering", "Koi_Fish",
    "Low_Frequency_Burst", "Scattered_Light", "Tomte", "Whistle",
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


## Fetch + whiten helpers (identical to `validate_gspy_o3.py`)

In [3]:
def whiten(ts):
    """Whiten and trim, using identical params to get_clean_backgrounds.py."""
    w = ts.whiten(fftlength=PSD_FFTLENGTH, overlap=PSD_OVERLAP, highpass=HIGHPASS_HZ)
    pad = MAX_FILTER_DUR * SAMPLE_RATE
    arr = np.array(w, dtype=np.float32)[pad:-pad]
    np.clip(arr, -MAX_AMP, MAX_AMP, out=arr)
    return arr


def fetch_raw(ifo, start, end, data_source=DATA_SOURCE):
    if data_source == "open":
        return TimeSeries.fetch_open_data(ifo, start, end, sample_rate=SAMPLE_RATE)
    urls = find_urls(ifo[0], FRAME_TYPE[ifo], start, end)
    ts = TimeSeries.read(urls, channel=f"{ifo}:GDS-CALIB_STRAIN", start=start, end=end)
    if ts.sample_rate.value != SAMPLE_RATE:
        ts = ts.resample(SAMPLE_RATE)
    return ts


def fetch_whitened(ifo, gps_center, data_source=DATA_SOURCE):
    """Returns (full_white, central_4s, t0_gps) — t0_gps is full_white's true GPS start
    (raw fetch t0 + MAX_FILTER_DUR trimmed off the front). Using the raw fetch's t0
    directly here would misalign the injected residual from where GravitySpy looks."""
    half = FETCH_DUR // 2
    ts = fetch_raw(ifo, gps_center - half, gps_center + half, data_source)
    t0_gps = float(ts.t0.value) + MAX_FILTER_DUR
    white = whiten(ts)
    c, h = len(white) // 2, LEN_4S // 2
    if c - h < 0 or c + h > len(white):
        raise ValueError(f"Whitened segment too short ({len(white)}) after trim")
    return white, white[c - h : c + h].copy(), t0_gps

## DeepExtractor model + MC-Dropout extraction

In [4]:
def load_de_model(ckpt_path, dropout_p, device):
    model = UNET2D(in_channels=2, out_channels=2, dropout_p=dropout_p).to(device)
    load_checkpoint(torch.load(ckpt_path, map_location=device), model)
    model.eval()
    enable_mc_dropout(model)
    return model


def extract_glitch(model, scaler, whitened_4s, window, istft_fn, device, n_passes):
    """Returns (background, residual). residual = whitened_4s − background."""
    scaled = scaler.transform(whitened_4s.reshape(-1, 1)).astype(np.float32).reshape(1, -1)
    x_stft = apply_stft(scaled, N_FFT, HOP_LENGTH, WIN_LENGTH, window.cpu()).to(device)
    passes = mc_predict(model, x_stft, n_passes=n_passes, postprocess_fn=istft_fn)
    passes_np = passes.cpu().numpy().squeeze(1)
    bg_passes = scaler.inverse_transform(
        passes_np.reshape(-1, 1)
    ).reshape(passes_np.shape[0], -1).astype(np.float32)
    background = np.median(bg_passes, axis=0)
    residual = whitened_4s - background
    return background, residual


print(f"Loading DeepExtractor model: {CHECKPOINT}")
de_model = load_de_model(CHECKPOINT, DROPOUT_P, device)
with open(SCALER, "rb") as fh:
    scaler = pickle.load(fh)
window = torch.hann_window(WIN_LENGTH).to(device)
istft_fn = functools.partial(
    apply_istft, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, window=window,
)

Loading DeepExtractor model: /home/tom.dooney/deepextractor/checkpoints_4s_o3_mcd/DeepExtractor_257_checkpoints/checkpoint_best_o3_mcd_tl.pth.tar


FileNotFoundError: [Errno 2] No such file or directory: '/home/tom.dooney/deepextractor/checkpoints_4s_o3_mcd/DeepExtractor_257_checkpoints/checkpoint_best_o3_mcd_tl.pth.tar'

In [5]:
def compute_snr(glitch, srate=SAMPLE_RATE):
    """Optimal SNR of a whitened-frame (flat-PSD) signal against itself —
    same convention as deepextractor.utils.signal.snr_scaling with psd=None.
    A perfectly-reconstructed residual should read close to the CSV's own
    reported "snr" for that glitch; a much lower value means DeepExtractor
    is losing most of the glitch's power during separation, leaving a
    residual too weak for GravitySpy to recognize (independent of any
    other pipeline bug).
    """
    glitch = np.asarray(glitch)
    df = srate / glitch.shape[-1]
    glitch_fd = np.fft.rfft(glitch, axis=-1) / srate
    power = (np.conj(glitch_fd) * glitch_fd).real
    true_sigma_sq = 4.0 * df * np.sum(power, axis=-1)
    return float(np.sqrt(true_sigma_sq))

## Sample real O3 glitches (3 per class, H1 only, high-confidence GravitySpy labels)

In [ ]:
rng = np.random.default_rng(SEED)

df = pd.concat([pd.read_csv(p) for p in CSV_PATHS], ignore_index=True)
df = df[
    df["label"].isin(LABEL_ORDER)
    & (df["ifo"] == IFO)
    & (df["snr"] >= MIN_SNR)
].reset_index(drop=True)

samples_by_class = {}
for cls in LABEL_ORDER:
    sub = df[df["label"] == cls]
    n = min(N_PER_CLASS, len(sub))
    if n < N_PER_CLASS:
        print(f"  WARNING: only {n} samples for {cls}")
    idx = rng.choice(len(sub), size=n, replace=False)
    samples_by_class[cls] = sub.iloc[sorted(idx)].reset_index(drop=True)
    print(f"  {cls}: {n} samples")

## Extract residual glitches

In [ ]:
records = []
for cls in LABEL_ORDER:
    print(f"\n── {cls} ──")
    for _, row in tqdm(samples_by_class[cls].iterrows(), total=len(samples_by_class[cls]), desc=cls):
        gps = float(row["GPStime"])
        ifo = row["ifo"]
        try:
            _, whitened_4s, _ = fetch_whitened(ifo, gps)
        except Exception as e:
            print(f"  SKIP {ifo} GPS={gps:.3f}: {e}")
            continue
        with torch.no_grad():
            background, residual = extract_glitch(
                de_model, scaler, whitened_4s, window, istft_fn, device, N_PASSES
            )
        catalog_snr  = float(row["snr"])
        residual_snr = compute_snr(residual)
        records.append({
            "true_label": cls, "ifo": ifo, "gps": gps, "residual": residual,
            "catalog_snr": catalog_snr, "residual_snr": residual_snr,
        })
        print(f"  extracted  {cls:22s} {ifo}  GPS={gps:.3f}  catalog_snr={catalog_snr:6.1f}  residual_snr={residual_snr:6.1f}")

print(f"\n{len(records)} residuals extracted.")

Save the extracted residuals to disk (same format as `validate_gspy_o3.py`'s `residuals.npz`) so classification can be re-run later without redoing the real-data fetch + MC-Dropout extraction step.

In [ ]:
residuals_path = os.path.join(OUTPUT_DIR, "residuals.npz")
np.savez(
    residuals_path,
    true_label = np.array([r["true_label"] for r in records]),
    ifo        = np.array([r["ifo"]        for r in records]),
    gps        = np.array([r["gps"]        for r in records]),
    residual   = np.stack([r["residual"]   for r in records]),  # (N, LEN_4S)
)
print(f"Saved residuals → {residuals_path}")

### (Optional) Reload previously-saved residuals instead of re-extracting

Run **this cell instead of** the two extraction cells above if you already have a `residuals.npz` from a previous run and just want to re-debug the classification step (e.g. after changing `CLEAN_GPS_H1`, `GSPY_MODEL`, or GravitySpy itself) without redoing the real-data fetch + MC-Dropout extraction.

In [ ]:
residuals_path = os.path.join(OUTPUT_DIR, "residuals.npz")
data = np.load(residuals_path, allow_pickle=True)
records = [
    {"true_label": str(t), "ifo": str(i), "gps": float(g), "residual": r}
    for t, i, g, r in zip(
        data["true_label"], data["ifo"], data["gps"], data["residual"]
    )
]
print(f"Loaded {len(records)} residuals from {residuals_path}")

## Fetch the clean injection background

One clean, vetted-quiet real-noise segment for the configured IFO. Each residual glitch
gets injected into the **middle 4s** of this same segment before classification, since
GravitySpy expects realistic strain context rather than a bare isolated array.

In [ ]:
print("Fetching clean injection background...")
clean_gps = {"H1": CLEAN_GPS_H1}[IFO]
bg_white, _central_4s, bg_t0 = fetch_whitened(IFO, clean_gps)
print(f"  {IFO} @ GPS {clean_gps}  (len={len(bg_white)} samples, t0={bg_t0})")

## Classify every residual with GravitySpy

**Verbose** — prints true label, predicted label, and confidence for every single sample
as it's classified, so you can see the exact prediction pattern (e.g. always predicting
one class, or scattered wrong guesses) rather than only the aggregated confusion matrix.

In [ ]:
from gravityspy.classify import classify as gspy_classify
# Matches glitchgan/notebooks/gspy_classification.ipynb's import block exactly.
# Importing labelling_test_glitches runs its module-level
# K.set_image_data_format("channels_last") side effect — without this explicit
# import here, that side effect only fires whichever moment gravityspy's own lazy
# internal import chain happens to trigger it, which may be later than expected.
import gravityspy.ml.labelling_test_glitches as _lgt

warnings.filterwarnings("ignore")
for _log in ["gravityspy", "gwpy", "astropy", "tensorflow"]:
    logging.getLogger(_log).setLevel(logging.ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

gspy_tmp = os.path.join(OUTPUT_DIR, "gspy_tmp")
shutil.rmtree(gspy_tmp, ignore_errors=True)
os.makedirs(gspy_tmp, exist_ok=True)



### Single-sample debug: one Blip glitch, step by step

Before looping over all 21 residuals, walk through exactly one — inject it into the
clean background, plot the 4s window GravitySpy will actually see (with the injected
glitch overlaid on the input), Q-scan that same 4s window, then classify just this
one sample.

In [ ]:
sample_idx = next(i for i, r in enumerate(records) if r["true_label"] == "Blip")
rec = records[sample_idx]
ifo = rec["ifo"]
residual = rec["residual"]
if SNR_TARGET is not None:
    residual = whitened_snr_scaling(residual, SNR_TARGET, SAMPLE_RATE)

injected = bg_white.copy()
c, h = len(injected) // 2, LEN_4S // 2
injected[c - h : c + h] += residual
injected_4s = injected[c - h : c + h]   # the exact 4s window the glitch sits in

# Plot: 4s of background+glitch (the actual GravitySpy input), with the residual overlaid
t = np.linspace(0, 4, LEN_4S)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, injected_4s, color="grey", lw=0.8, alpha=0.6, label="Background + injected glitch")
ax.plot(t, residual, "r--", lw=1.2, label="Injected residual (Blip)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Whitened strain")
ax.set_title(f"Single-sample debug — Blip  (H1 GPS={rec['gps']:.3f})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Q-scan of the same 4s window — this is what GravitySpy is actually classifying
fig, ax = plt.subplots(figsize=(8, 4))
plot_q_transform(
    injected_4s, srate=SAMPLE_RATE, whiten=False,
    qrange=[10, 10], frange=[10, 1200],
    ax=ax, colourbar=True,
)
ax.set_title(f"Q-scan — Blip injected sample  (H1 GPS={rec['gps']:.3f})", fontsize=11)
plt.tight_layout()
plt.show()

# Classify just this one sample
ts_single = TimeSeries(injected, t0=bg_t0, sample_rate=SAMPLE_RATE, name=ifo)
result = gspy_classify(
    event_time=clean_gps,
    channel_name=f"{ifo}:GDS-CALIB_STRAIN",
    path_to_cnn=GSPY_MODEL,
    timeseries=ts_single,
    plot_directory=gspy_tmp,
    whiten=False,
)
pred = result["ml_label"].value[0]
conf = float(result["ml_confidence"].value[0])
print(f"true=Blip  pred={pred}  conf={conf:.3f}")

In [ ]:
rows = []
for i, rec in enumerate(records):
    ifo = rec["ifo"]
    residual = rec["residual"]
    if SNR_TARGET is not None:
        residual = whitened_snr_scaling(residual, SNR_TARGET, SAMPLE_RATE)

    injected = bg_white.copy()
    c, h = len(injected) // 2, LEN_4S // 2
    injected[c - h : c + h] += residual

    ts = TimeSeries(injected, t0=bg_t0, sample_rate=SAMPLE_RATE, name=ifo)
    try:
        result = gspy_classify(
            event_time=clean_gps,
            channel_name=f"{ifo}:GDS-CALIB_STRAIN",
            path_to_cnn=GSPY_MODEL,
            timeseries=ts,
            plot_directory=gspy_tmp,
            # ts is already gwpy-whitened (see fetch_whitened). NOTE: as of the
            # installed gravityspy, this kwarg is currently a no-op — make_q_scans
            # hardcodes whiten=True directly in its q_transform() calls and never
            # reads a 'whiten' key out of **kwargs. Left here to document intent.
            whiten=False,
        )
        pred = result["ml_label"].value[0]
        conf = float(result["ml_confidence"].value[0])
    except Exception as e:
        print(f"  GravitySpy error {ifo} {rec['gps']:.3f}: {e}")
        pred, conf = "Error", 0.0

    mark = "OK   " if pred == rec["true_label"] else "WRONG"
    print(f"[{i+1:2d}/{len(records)}] {mark}  true={rec['true_label']:22s}  pred={pred:22s}  conf={conf:.3f}  ({ifo} GPS={rec['gps']:.3f})")

    rows.append({
        "true_label": rec["true_label"], "pred_label": pred,
        "confidence": conf, "ifo": ifo, "gps": rec["gps"],
    })

df_res = pd.DataFrame(rows)
csv_out = os.path.join(OUTPUT_DIR, "gspy_results.csv")
df_res.to_csv(csv_out, index=False)
print(f"\nSaved {csv_out}")

## Quick diagnostics

In [ ]:
print("Predicted label counts:")
print(df_res["pred_label"].value_counts())
print()
print("Mean confidence:", df_res["confidence"].mean())
print()
print("Per-true-label accuracy:")
for cls in LABEL_ORDER:
    sub = df_res[df_res["true_label"] == cls]
    if len(sub) == 0:
        continue
    acc = (sub["pred_label"] == cls).mean()
    print(f"  {cls:22s}  n={len(sub)}  acc={acc:.2f}")

## Confusion matrix

In [ ]:
def plot_confusion(df_res, run_label):
    df_valid = df_res[df_res["pred_label"] != "Error"].copy()
    if df_valid.empty:
        print("No valid GravitySpy results — skipping confusion matrix.")
        return None

    pred_all = sorted(df_valid["pred_label"].unique())
    pred_cols = [l for l in LABEL_ORDER if l in pred_all] + [l for l in pred_all if l not in LABEL_ORDER]
    count_matrix = pd.DataFrame(0, index=LABEL_ORDER, columns=pred_cols)
    conf_acc = {(t, p): [] for t in LABEL_ORDER for p in pred_cols}
    for _, row in df_valid.iterrows():
        t, p, c = row["true_label"], row["pred_label"], row["confidence"]
        if t in LABEL_ORDER and p in pred_cols:
            count_matrix.loc[t, p] += 1
            conf_acc[(t, p)].append(c)

    annot = pd.DataFrame("", index=LABEL_ORDER, columns=pred_cols)
    for t in LABEL_ORDER:
        for p in pred_cols:
            n = count_matrix.loc[t, p]
            annot.loc[t, p] = "0" if n == 0 else f"{n}\n({np.mean(conf_acc[(t, p)]):.2f})"

    total = count_matrix.values.sum()
    acc = np.trace(count_matrix.values) / total if total > 0 else 0.0

    fig, ax = plt.subplots(figsize=(max(10, len(pred_cols) * 1.1), 6))
    sns.heatmap(count_matrix, annot=annot, fmt="", cmap="Blues",
                linewidths=0.5, linecolor="gray",
                annot_kws={"size": 8, "color": "black"}, ax=ax)
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(f"GravitySpy — 4s DeepExtractor {run_label} TL  (accuracy = {acc:.1%})", fontsize=13)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix.{ext}"), dpi=150, bbox_inches="tight")
    print(f"Confusion matrix — accuracy: {acc:.3f}  ({total} samples)")
    return fig


plot_confusion(df_res, RUN_LABEL)